# Pocket Pet: ternary inference to FPGA handoff
This notebook executes a tiny offline transformer. It validates mechanics and memory accounting, not pretrained language quality. No model download is required.

In [ ]:
import json

import torch

from pocket_pet import PocketPetConfig, PocketPetTransformer
from pocket_pet.model import benchmark_model
from pocket_pet.quantization import ternary_decompose

torch.manual_seed(7)
config = PocketPetConfig(
    vocab_size=128,
    hidden_size=64,
    intermediate_size=160,
    num_layers=2,
    num_heads=4,
    latent_size=16,
    max_sequence_length=128,
)
model = PocketPetTransformer(config).eval()
print(f"{sum(p.numel() for p in model.parameters()):,} trainable parameters")

## 1. Inspect the frozen weight representation
The bulk path contains only -1, 0, and +1. A fixed fraction of large-magnitude values bypasses it exactly. The decomposition is the software reference for a packed FPGA image.

In [ ]:
layer = model.layers[0].attention.q_proj
symbols, scales, protected, mask = ternary_decompose(layer.weight, config.outlier_fraction)
print("symbols:", symbols.unique().tolist())
print("protected weights:", int(mask.sum()), "/", mask.numel())
print(json.dumps(model.storage_report(), indent=2))

## 2. Run causal generation and measure writable memory
The random model produces meaningless tokens, but every token is computed through causal attention. The cache contains learned latents quantized to int8 plus FP16 scales.

In [ ]:
prompt = torch.tensor([[1, 8, 3, 5]])
generated, cache = model.generate(prompt, max_new_tokens=8)
print("token ids:", generated.tolist()[0])
print("latent cache bytes:", cache.memory_bytes())
print("equivalent FP16 full K/V bytes:", cache.fp16_full_kv_bytes(config))
print("compression:", round(cache.fp16_full_kv_bytes(config) / cache.memory_bytes(), 2), "x")

In [ ]:
report = benchmark_model(model, prompt_length=16, new_tokens=4, repeats=1)
print(json.dumps(report, indent=2))

## 3. Project cache scaling
Stage 1 varies latent width and sequence length while measuring quality. Stage 2 fixes one format and hands packed symbols, scales, protected streams, cache contracts, and golden vectors to RTL.

In [ ]:
import matplotlib.pyplot as plt

lengths = torch.tensor([1_000, 4_000, 16_000, 32_000, 120_000])
latent_mb = lengths * config.num_layers * (config.latent_size + 2) / 1e6
full_mb = lengths * config.num_layers * 2 * config.hidden_size * 2 / 1e6
plt.figure(figsize=(7, 3.5))
plt.plot(lengths, full_mb, "o-", label="conventional FP16 K/V")
plt.plot(lengths, latent_mb, "o-", label="int8 latent + FP16 scale")
plt.xlabel("context tokens")
plt.ylabel("cache MB")
plt.title("Writable cache scaling (prototype dimensions)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## FPGA acceptance contract
1. Pack two-bit symbols in physical tile order; ship FP16 channel scales and sparse protected `(index, value)` streams.
2. Match golden layer activations before optimizing accumulator width.
3. Append int8 latent bytes and FP16 scales via DMA; fuse expansion with tiled attention.
4. Gate on causal correctness, bounded numerical error, tokens/s, time-to-first-token, joules/token, and DRAM bytes/token.